In [1]:
print("AI with Prasant")

AI with Prasant


In [2]:
!pip install unsloth # install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git # Also get the latest version Unsloth!

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 689.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/92

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-7s_aexg5
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-7s_aexg5
  Resolved https://github.com/unslothai/unsloth.git to commit 2554636ded46f87430cac6c20a35c828801585c8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.6.1-py3-none-any.whl size=35399300 sha256=bec87ad49ffbad20751c2e9fd7b79834dc296b266ad41fa6f6f881602ac7f86f
  Stored in directory: /tmp/pip-ephem-wheel-cache-kgafgg71/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2026.6.1
    Uninstalling unsloth-2026.6.1:
      Successfully uninstalled unsloth-2026.6.1


In [1]:
 #Step3: Import necessary libraries
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from huggingface_hub import login
from transformers import TrainingArguments
from datasets import load_dataset
import wandb

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# Check HF token
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [3]:
# Optional: Checking GPU availability
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0)) if torch.cuda.is_available() else "No GPU"


CUDA available: True
GPU device: Tesla T4


In [5]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
max_sequence_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_sequence_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token
)


==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


TimeoutError: Unsloth: HuggingFace seems to be down after trying for 120 seconds :(
Check https://status.huggingface.co/ for more details.
As a temporary measure, use modelscope with the same model name ie:
```
pip install modelscope
import os; os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
from unsloth import FastLanguageModel
model = FastLanguageModel.from_pretrained('unsloth/gpt-oss-20b')
```

In [5]:
#Setup system prompt
prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Below crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response in both and accurate.

### Instruction:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Question:
{}

### Response:
<think>{}
"""

In [6]:
# Run Inference on the model

# Define a test question
question = """ A 61-year-old woman with a long history of involuntary urine loss duing activities like coughing or sneezing but no leakage at night undergoes a gynecological eaxm and Q-tip test. Based on these findings,what would cystometry most likely reveal about her residual volume and detrusor contractions.
"""

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200, # Corrected from max_new_token to max_new_tokens
    use_cache = True
)

# Decode the response tokens back to text
inputs = tokenizer(...)
outputs = model.generate(...)
response = tokenizer.batch_decode(outputs)
print(response)

Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


RuntimeError: 
🚨 CUDA SETUP ERROR: Missing dependency: libnvJitLink.so.13 🚨

CUDA 13.x runtime libraries were not found in the LD_LIBRARY_PATH.

To fix this, make sure that:
1. You have installed CUDA 13.x toolkit on your system
2. The CUDA runtime libraries are in your LD_LIBRARY_PATH

You can add them with (and persist the change by adding the line to your .bashrc):
   export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/path/to/cuda-13.x/                    lib64

Original error: libnvJitLink.so.13: cannot open shared object file: No such file or directory

🔍 Run this command for detailed diagnostics:
python -m bitsandbytes

If you've tried everything and still have issues:
1. Include ALL version info (operating system, bitsandbytes, pytorch, cuda, python)
2. Describe what you've tried in detail
3. Open an issue with this information:
   https://github.com/bitsandbytes-foundation/bitsandbytes/issues

Native code method attempted to call: lib.cdequantize_blockwise_fp32()

In [ ]:
clean_response = response[0] # Get the first (and likely only) response string

# Find the actual answer part using the correct separator
answer_start_tag = "###Answer:<think>"
if answer_start_tag in clean_response:
    clean_response = clean_response.split(answer_start_tag, 1)[1]
else:
    # Fallback if the tag is not found, or adjust based on desired behavior
    clean_response = clean_response # Keep original if tag not found or handle as error

# Replace Unsloth's custom spacing tokens and newlines
clean_response = clean_response.replace("Ġ", " ")
clean_response = clean_response.replace("Ċ", "\n")

# Optionally remove any remaining special tokens if present at the start/end
clean_response = clean_response.replace("<|begin of sentence|>", "").replace("<|end of sentence|>", "").strip()

print(clean_response)

In [ ]:
# setup fine-tuning

# Load Dataset
medical_dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT","en",split = "train[:500]", trust_remote_code = True)

In [ ]:
medical_dataset[1]

In [ ]:
EOS_TOKEN = tokenizer.eos_token
EOS_TOKEN
# Define EOS_TOKEN (End Of Sentence) which tells the model when to stop generating text during training

In [ ]:
# Update the prompt_style for training

train_prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Below crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response in both and accurate.

### Instruction:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

In [ ]:
from prompt_toolkit import output
# Prepare the data for fine-tuning

def preprocess_input_data(examples):
  inputs = examples["Question"]
  cots = examples["Complex_CoT"]
  outputs = examples["Response"]

  texts = []
  for input, cot, output in zip(inputs, cots, outputs):
    text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
    texts.append(text)

  return {
      "texts" : texts,
  }

In [ ]:
finetune_dataset = medical_dataset.map(
    preprocess_input_data,
    batched=True
)

In [ ]:
finetune_dataset["texts"][1]

In [ ]:
# Setup/Apply LoRA finetuning to the model

model_lora = FastLanguageModel.get_peft_model(
    model = model,
    r = 16, # New adepter size 16 which is going to trained
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    use_rslora = False,
    loftq_config = None
)

In [ ]:
# Add this before creating the trainer
if hasattr(model, '_unwrapped_old_generate'):
  del model._unwrapped_old_generate

In [ ]:
print(type(finetune_dataset["texts"][0]))
print(finetune_dataset["texts"][0][:500])

In [ ]:
print(finetune_dataset["texts"][0][-1000:])

In [ ]:
# Define the formatting function as required by Unsloth's SFTTrainer
def formatting_prompts_func(examples):
    # `examples["texts"]` will be a list of lists, where each inner list contains one formatted string.
    # We need to flatten this into a single list of strings.
    flattened_texts = [text for sublist in examples["texts"] for text in sublist]
    return flattened_texts

trainer = SFTTrainer(
    model = model_lora,
    tokenizer = tokenizer,
    train_dataset = finetune_dataset,
    # The 'dataset_text_field' is not used when 'formatting_func' is provided.
    # It seems Unsloth's SFTTrainer explicitly requires 'formatting_func' in this context.
    formatting_func = formatting_prompts_func, # Pass the formatting function
    max_seq_length = max_sequence_length,
    dataset_num_proc = 1,

    # Define training args
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir = "outputs",
        gradient_checkpointing = False # Explicitly set gradient_checkpointing to False
    )
)

In [ ]:
# Setup, WANDB
from google.colab import userdata
wnb_token = userdata.get("WANDB_API_TOKEN")
#login to WnB
wandb.login(key=wnb_token) #import wandb
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-on-Medical-CoT-Dataset',
    job_type="training",
    anonymous="allow"
)


In [ ]:
import torch

# Start the fine-tuning process
trainer_stats = trainer.train()

In [ ]:
wandb.finish()

In [ ]:
# Testing after finetuning
# Define a test question
question = """ A 61-year-old woman with a long history of involuntary urine loss duing activities like coughing or sneezing but no leakage at night undergoes a gynecological eaxm and Q-tip test. Based on these findings,what would cystometry most likely reveal about her residual volume and detrusor contractions.
"""
# Ensure FastLanguageModel is defined in this scope
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model_lora)

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model_lora.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200, # Corrected from max_new_token to max_new_tokens
    use_cache = True
)

# Decode the response tokens back to text
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

In [ ]:
print(response.split("###Response:")[1])

In [ ]:
print(tokenizer.__class__)
print(tokenizer.name_or_path)